# 🔧 Notebook 2 — Data Preprocessing
**SoilHealth Predictor: Machine Learning for Soil Fertility Classification**  
SRM Institute of Science and Technology | Data Science Mini Project 2026

---

### Objectives
- Clean the raw dataset (handle outliers, verify types)
- Encode target labels with `LabelEncoder`
- Normalize features with `StandardScaler`
- Perform stratified train-test split (80/20)
- Save processed files for model training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys

from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

sys.path.append(os.path.join('..', 'src'))
FEATURES = ['N', 'P', 'K', 'pH', 'Moisture', 'Temperature', 'EC']
CLASS_ORDER = ['High Fertility', 'Medium Fertility', 'Low Fertility', 'Poor Fertility']
RANDOM_SEED = 42

sns.set_theme(style='whitegrid')
print('Ready ✓')

## 1. Load Raw Data

In [ ]:
raw_path = os.path.join('..', 'dataset', 'raw_data', 'soil_data.csv')
df = pd.read_csv(raw_path)
print(f'Loaded: {df.shape}')
df.head()

## 2. Outlier Detection (IQR Method)

In [ ]:
print('Outlier Report (IQR method):')
outlier_counts = {}
for col in FEATURES:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    outlier_counts[col] = count
    pct = count / len(df) * 100
    print(f'  {col:<14}: {count:3d} outliers ({pct:.1f}%)')

print('\nOutliers are retained — they may represent genuine extreme field conditions.')

In [ ]:
# Visualise outlier counts
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(outlier_counts.keys(), outlier_counts.values(), color='#3B6D11', edgecolor='none')
ax.set_ylabel('Outlier Count')
ax.set_title('IQR-Based Outlier Counts per Feature', fontweight='bold')
ax.grid(axis='y', linewidth=0.4)
plt.tight_layout()
plt.show()

## 3. Label Encoding

In [ ]:
le = LabelEncoder()
le.fit(CLASS_ORDER)   # fit in desired order
df['Label'] = le.transform(df['Fertility_Class'])

print('Label Encoding Map:')
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {code}  →  {cls}')

df[['Fertility_Class', 'Label']].value_counts().sort_index()

## 4. Train-Test Split (Stratified, 80/20)

In [ ]:
X = df[FEATURES].values
y = df['Label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f'Training set  : {X_train.shape[0]} samples')
print(f'Test set      : {X_test.shape[0]} samples')

# Verify stratification
from collections import Counter
train_dist = {le.classes_[k]: v for k, v in sorted(Counter(y_train).items())}
test_dist  = {le.classes_[k]: v for k, v in sorted(Counter(y_test).items())}
print(f'\nTrain distribution: {train_dist}')
print(f'Test  distribution: {test_dist}')

## 5. Feature Scaling (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit ONLY on train
X_test_s  = scaler.transform(X_test)       # transform test using train params

print('Scaling parameters (per feature):')
for feat, mean, std in zip(FEATURES, scaler.mean_, scaler.scale_):
    print(f'  {feat:<14}  mean={mean:8.3f}   std={std:.3f}')

In [ ]:
# Compare before/after scaling for one feature
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(X_train[:, 0], bins=30, color='#97C459', edgecolor='none')
axes[0].set_title('N (raw) — before scaling', fontweight='bold')
axes[0].set_xlabel('mg/kg')

axes[1].hist(X_train_s[:, 0], bins=30, color='#3B6D11', edgecolor='none')
axes[1].set_title('N (scaled) — after StandardScaler', fontweight='bold')
axes[1].set_xlabel('z-score')

plt.suptitle('Effect of StandardScaler on Nitrogen Feature', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Save Processed Data

In [ ]:
proc_dir = os.path.join('..', 'dataset', 'processed_data')
os.makedirs(proc_dir, exist_ok=True)

train_df = pd.DataFrame(X_train_s, columns=FEATURES)
train_df['Label'] = y_train
train_df.to_csv(os.path.join(proc_dir, 'train.csv'), index=False)

test_df = pd.DataFrame(X_test_s, columns=FEATURES)
test_df['Label'] = y_test
test_df.to_csv(os.path.join(proc_dir, 'test.csv'), index=False)

print('Saved: dataset/processed_data/train.csv')
print('Saved: dataset/processed_data/test.csv')

---
### Summary
| Step | Action | Result |
|---|---|---|
| Null check | No nulls found | ✓ Clean |
| Outlier check | IQR method, retained | ✓ Noted |
| Label encoding | LabelEncoder (4 classes) | 0–3 |
| Train-test split | 80/20, stratified | 1760 / 440 |
| Scaling | StandardScaler on train only | z-scores |

➡️ Proceed to **Notebook 03 — Visualization**